# Group theory 7: Group actions and counting

*Learning path: group theory, part 7 of 10.* [Index](README.md) · Previous: [Galois and symmetry](06-galois-and-symmetry.ipynb) · Next: [Polynomials over fields](08-polynomials-over-fields.ipynb)

**The question.** How many different necklaces can you make from six beads, each black or white? There are
$2^6 = 64$ ways to colour six beads in a row, but a necklace can be turned, so many of those 64 are the same
necklace. And a chemist asks the same kind of question: if two chlorine atoms replace two hydrogens on a benzene ring,
how many different molecules can result? Counting things "up to symmetry" is what group actions are for, and one
short formula, Burnside's lemma, answers both questions.

**Who this is for:** anyone who has done lessons 1 to 4. **You need:** permutations, `group`, `order` and
`elements`, and Lagrange's theorem.

**You will learn**

* what it means for a group to **act** on a set, with rotations acting on beads, vertices and faces,
* **orbits** and **stabilizers**, and the orbit–stabilizer theorem $|G| = |\text{orbit}|\cdot|\text{stabilizer}|$,
* **Burnside's lemma**: the number of orbits is the average number of fixed points,
* how to count necklaces, bracelets, coloured cubes and chemical isomers with it.

## 1. Groups that move things

A group **acts** on a set when each element of the group moves the points of the set around, the identity moves
nothing, and doing $g$ then $h$ is the same as doing the product. The groups of lessons 1 to 4 were born acting: a
permutation in `perm` is a rule that sends point $i$ to point $p(i)$.

Here are the rotations of a cube, acting on its 8 corners. Number the top face 1, 2, 3, 4 going round, and the bottom
face 5, 6, 7, 8 underneath them. A quarter turn about the vertical axis and a quarter turn about a horizontal axis
generate all of them:

In [1]:
cube = group(perm("(1 2 3 4)(5 6 7 8)", 8), perm("(1 2 6 5)(4 3 7 8)", 8));
order(cube)

24

> `order`: order(G) is the number of elements of G.

24 rotations: the identity, 9 quarter and half turns about the 3 axes through opposite faces, 8 turns of a third about
the 4 long diagonals, and 6 half turns about axes through opposite edges ($1 + 9 + 8 + 6 = 24$).

Where can a group element send corner 1? Collect $g(1)$ over the whole group. That set is the **orbit** of 1. The
elements that leave 1 where it is form its **stabilizer**:

In [2]:
function orbit(G, x)
  E = elements(G);
  o = [];
  for r in 1:order(G)
    g = E(r);
    if !contains(o, g(x))
      o = append(o, g(x));
    end
  end
  return o
end
function stabilizer(G, x)
  E = elements(G);
  c = 0;
  for r in 1:order(G)
    g = E(r);
    if g(x) == x
      c = c + 1;
    end
  end
  return c
end
orbit(cube, 1)
stabilizer(cube, 1)

[1, 2, 3, 4, 5, 6, 7, 8]
3

Every corner can reach every other, and 3 rotations keep corner 1 in place: the identity and the two turns about the
diagonal through it. And $8 \times 3 = 24$.

## 2. The orbit–stabilizer theorem

That was no accident. If $g$ and $h$ both send $x$ to the same point $y$, then $h^{-1}g$ fixes $x$, so $g$ and $h$
lie in the same coset of the stabilizer. Each point of the orbit corresponds to exactly one coset, and by Lagrange's
theorem (lesson 3) all cosets have the same size:

$$|G| = |\text{orbit of } x| \cdot |\text{stabilizer of } x|.$$

Use it in reverse to count a symmetry group without listing it. The same 24 rotations act on the cube's 6 faces
(1 top, 2 to 5 round the sides, 6 bottom): every face can be turned to the top, and 4 rotations keep the top on top.

In [3]:
faces = group(perm("(2 3 4 5)", 6), perm("(1 2 6 4)", 6));
[order(faces), numel(orbit(faces, 1)), stabilizer(faces, 1)]

[24, 6, 4]

$6 \times 4 = 24$ again. The same argument counts the rotations of a soccer ball (a truncated icosahedron): 12 pentagons
can be brought to the top, and 5 rotations keep a pentagon there, so it has $12 \times 5 = 60$.

**Try this.** A tetrahedron has 4 faces, and 3 rotations keep a face in place. How many rotations does it have? Build
it as `group(perm("(1 2 3)", 4), perm("(1 2)(3 4)", 4))` acting on its corners and check.

## 3. Burnside's lemma

Back to the necklaces. The rotations of a six-bead necklace form the cyclic group of order 6, acting on the 64
colourings. Two colourings are the same necklace exactly when they lie in the same orbit, so we want the **number of
orbits**. Burnside's lemma (in fact Cauchy's, 1845, and Frobenius's, 1887) says

$$\#\text{orbits} = \frac{1}{|G|}\sum_{g \in G}\#\{\text{colourings fixed by } g\}.$$

(Count pairs ($g$, colouring) with $g$ fixing the colouring in two ways: by orbit-stabilizer, each orbit contributes
exactly $|G|$ pairs.)

A colouring is fixed by $g$ when every cycle of $g$ is one colour, so with $k$ colours $g$ fixes $k^{c(g)}$ colourings,
where $c(g)$ counts the cycles of $g$, fixed points included:

In [4]:
function ncycles(p)
  n = numel(p);
  seen = zeros(1, n);
  c = 0;
  for s in 1:n
    if seen(s) == 0
      c = c + 1;
      j = s;
      while seen(j) == 0
        seen(j) = 1;
        j = p(j);
      end
    end
  end
  return c
end
function burnside(G, k)
  E = elements(G);
  total = 0;
  for r in 1:order(G)
    total = total + k^ncycles(E(r));
  end
  return total/order(G)
end
C6 = group(perm("(1 2 3 4 5 6)", 6));
burnside(C6, 2)

14

14 necklaces. The six rotations fix $2^6, 2^1, 2^2, 2^3, 2^2, 2^1$ colourings (the turn by one bead has one cycle of
6; the half turn has three cycles of 2), and $(64 + 2 + 4 + 8 + 4 + 2)/6 = 14$.

If the necklace can also be flipped over (a **bracelet**), the group is the dihedral group of order 12:

In [5]:
D6 = group(perm("(1 2 3 4 5 6)", 6), perm("(2 6)(3 5)", 6));
burnside(D6, 2)

13

13: exactly one pair of necklaces are mirror images of each other and not turnings of each other. With 3 colours:

In [6]:
[burnside(C6, 3), burnside(D6, 3)]

[130, 92]

For a cycle of $n$ beads the rotation by $j$ steps has $\gcd(j, n)$ cycles, and grouping the rotations by that gcd gives
a closed formula, $\frac1n\sum_{d \mid n}\varphi(d)\,k^{n/d}$:

In [7]:
function necklaces(n, k)
  s = 0;
  for d in divisors(n)
    s = s + eulerphi(d)*k^(n/d);
  end
  return s/n
end
table([n, necklaces(n, 2), 2^n], n, 1, 12)

[ 1,   2,    2;
  2,   3,    4;
  3,   4,    8;
  4,   6,   16;
  5,   8,   32;
  6,  14,   64;
  7,  20,  128;
  8,  36,  256;
  9,  60,  512;
 10, 108, 1024;
 11, 188, 2048;
 12, 352, 4096]

> `table`: A row with expr for each k; when expr is itself a row, the rows stack into a matrix.

For large $n$ almost no colouring has any symmetry, so the number of necklaces approaches $2^n/n$.

**Try this.** Use `burnside(cube, 2)` and `burnside(faces, 3)`: how many ways are there to colour the corners of a
cube black or white, and the faces with three colours?

## 4. Isomers

On a benzene ring, six carbon atoms sit at the corners of a hexagon, each carrying a hydrogen. Replace two of the
hydrogens by chlorine. There are $\binom62 = 15$ ways to pick the two positions, but the molecule can be turned and
flipped, so we want the orbits of the dihedral group on pairs of positions. Give each pair a label, the smallest
pair it can be turned into, and collect the different labels:

In [8]:
function label(G, a, b)
  E = elements(G);
  best = 99;
  for r in 1:order(G)
    g = E(r);
    best = min(best, 10*min(g(a), g(b)) + max(g(a), g(b)));
  end
  return best
end
L = [];
for a in 1:5
  for b in a+1:6
    if !contains(L, label(D6, a, b))
      L = append(L, label(D6, a, b));
    end
  end
end
L

[12, 13, 14]

Three dichlorobenzenes: the chlorines next to each other (12, *ortho*), one apart (13, *meta*) or opposite (14,
*para*). Chemists have made all three, and only three: they boil at 180, 173 and 174 °C. In the 1870s, counting
isomers like these was one of the arguments for Kekulé's hexagon. Burnside agrees without listing them: the pairs of
positions are coloured with two colours, two of one and four of the other, and the weighted version of the lemma
(Pólya's theorem) counts exactly those.

**Try this.** How many trichlorobenzenes are there? (Adapt `label` to three positions, or count orbits of 3-subsets by
hand: 20 subsets in all.)

## Exercises

1. **Orbit sizes.** The rotations of a cube act on its 12 edges. What is the stabilizer of an edge, and what does
   orbit-stabilizer predict?
2. **Necklaces of 7.** Compute `necklaces(7, 2)`. Why is $2^7 - 2$ divisible by 7, and how does that explain the
   answer? (Fermat's little theorem from lesson 3 follows from this count.)
3. **Dice.** How many different dice can be made by putting the numbers 1 to 6 on the faces of a cube? (All $6!$
   labellings; only the identity fixes any of them.)
4. **Coloured squares.** Colour the 4 corners of a square with $k$ colours, up to rotations and reflections. Find
   the formula in $k$ with Burnside.
5. **Conjugation.** A group acts on itself by $g \cdot x = gxg^{-1}$. What are the orbits called (lesson 4)? Use
   orbit-stabilizer to show every class size divides $|G|$.
6. **Graphs.** How many graphs on 3 unlabelled vertices are there? (The symmetric group $S_3$ acts on the 3 possible
   edges; each edge is present or absent.)

## Summary

* A group acts on a set when its elements permute the points compatibly with the product.
* The **orbit** of $x$ is where $x$ can go; its **stabilizer** is what keeps it in place; $|G| = |\text{orbit}|\cdot|\text{stabilizer}|$.
* **Burnside's lemma**: the number of orbits is the average number of fixed points. A colouring is fixed by $g$ when
  each cycle of $g$ is one colour, so $g$ fixes $k^{c(g)}$ colourings.
* This counts necklaces (14 of six beads in two colours), bracelets (13), coloured cubes (57 with three colours) and
  isomers (3 dichlorobenzenes).

## Hints and answers

* **Section 2, Try this.** $4 \times 3 = 12$, and `order` of that group is 12.
* **Section 3, Try this.** 23 corner colourings; 57 face colourings.
* **Section 4, Try this.** Three: positions 123, 124 and 135.
* **Exercise 1.** The stabilizer of an edge has 2 elements (the identity and the half turn about the axis through its
  midpoint), so the orbit has 12 edges: all of them.
* **Exercise 2.** 20 necklaces. The 126 colourings that are not all one colour fall into orbits of exactly 7, so
  $7 \mid 2^7 - 2$, and $126/7 + 2 = 20$.
* **Exercise 3.** $720/24 = 30$.
* **Exercise 4.** $(k^4 + 2k + 3k^2 + 2k^3)/8$; for $k = 2$, 6 colourings.
* **Exercise 5.** Conjugacy classes; the size of a class is $|G|$ divided by the size of a stabilizer (the
  centralizer).
* **Exercise 6.** 4: no edges, one, two, or the triangle. Burnside: $(8 + 3\cdot4 + 2\cdot2)/6 = 4$.

## Sources

1. W. Burnside, *Theory of Groups of Finite Order*, Cambridge University Press (1897), §119.
2. G. Frobenius, "Über die Congruenz nach einem aus zwei endlichen Gruppen gebildeten Doppelmodul", *Journal für die
   reine und angewandte Mathematik* 101, 273–299 (1887).
3. G. Pólya, "Kombinatorische Anzahlbestimmungen für Gruppen, Graphen und chemische Verbindungen", *Acta Mathematica*
   68, 145–254 (1937).
4. P. M. Neumann, "A lemma that is not Burnside's", *The Mathematical Scientist* 4, 133–141 (1979).
5. M. A. Armstrong, *Groups and Symmetry*, Springer (1988), chapters 17 and 18.